In [1]:
import ee
from pathlib import Path
import shutil
import subprocess

ee.Initialize()

In [2]:
# Porto Alegre bounding box
poa = ee.Geometry.Rectangle([-51.30344, -30.26945, -51.018852, -29.932474])

COLLECTION_ID = 'LANDSAT/LC08/C02/T1_L2'
START_DATE = '2018-01-01'
END_DATE = '2025-01-01'

EXPORT_FOLDER = 'earthengine'
EXPORT_SCALE = 30
MAX_PIXELS = 1e13

MAX_ST_UNCERTAINTY_K = 2.0
MIN_CLOUD_DISTANCE_KM = 1.0


In [3]:
def mask_and_scale_lst(image):
    qa = image.select('QA_PIXEL')
    mask = (
        qa.bitwiseAnd(1 << 0).eq(0)
        .And(qa.bitwiseAnd(1 << 1).eq(0))
        .And(qa.bitwiseAnd(1 << 2).eq(0))
        .And(qa.bitwiseAnd(1 << 3).eq(0))
        .And(qa.bitwiseAnd(1 << 4).eq(0))
    )

    lst_c = (
        image.select('ST_B10')
        .multiply(0.00341802)
        .add(149.0)
        .subtract(273.15)
        .rename('LST_C')
    )

    st_qa_k = image.select('ST_QA').multiply(0.01)
    cloud_dist_km = image.select('ST_CDIST').multiply(0.01)
    quality_mask = st_qa_k.lte(MAX_ST_UNCERTAINTY_K).And(cloud_dist_km.gte(MIN_CLOUD_DISTANCE_KM))

    return lst_c.updateMask(mask).updateMask(quality_mask).copyProperties(image, image.propertyNames())

lst_col = (
    ee.ImageCollection(COLLECTION_ID)
    .filterBounds(poa)
    .filterDate(START_DATE, END_DATE)
    .filter(ee.Filter.eq('PROCESSING_LEVEL', 'L2SP'))
    .map(mask_and_scale_lst)
)

annual = lst_col.select('LST_C').mean().clip(poa).rename('annual_climatology_daytime_lst_c')
name = 'poa_annual_climatology_daytime_lst_c'

task = ee.batch.Export.image.toDrive(
    image=annual,
    description=name,
    folder=EXPORT_FOLDER,
    fileNamePrefix=name,
    region=poa,
    scale=EXPORT_SCALE,
    crs='EPSG:3857',
    maxPixels=MAX_PIXELS,
    fileFormat='GeoTIFF'
)
task.start()
print('Started Drive export:', task.id)

Started Drive export: RPZAH6IELKJFDGQAK3L6572A


### Convert to COG and Generate Tiles

In [4]:
base = Path('data')
in_tif = base / 'poa_annual_climatology_daytime_lst_c.tif'
out_dir = Path('out/annual_climatology_daytime_lst_c')
cog_tif = out_dir / 'poa_annual_climatology_daytime_lst_c_cog.tif'
visual_tiles_dir = out_dir / 'tiles_visual'
value_tiles_dir = out_dir / 'tiles_values'
colors_txt = base / 'annual_climatology_daytime_lst_c_colors.txt'

out_dir.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'gdal_translate', str(in_tif), str(cog_tif),
    '-of', 'COG', '-ot', 'Float32',
    '-co', 'COMPRESS=DEFLATE', '-co', 'RESAMPLING=NEAREST', '-co', 'OVERVIEWS=AUTO'
], check=True)

colorized_tif = out_dir / 'poa_annual_climatology_daytime_lst_c_colorized.tif'
subprocess.run(['gdaldem', 'color-relief', str(cog_tif), str(colors_txt), str(colorized_tif)], check=True)

visual_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['gdal2tiles.py', '-r', 'near', '-z', '8-15', '--xyz', '-w', 'none', str(colorized_tif), str(visual_tiles_dir)], check=True)
print('Visual tiles written to:', visual_tiles_dir)

# Encode value tiles: decode in app with lst_c = (R + 256*G + 65536*B - 10000) / 100
value_encoded_tif = out_dir / 'poa_annual_climatology_daytime_lst_c_value_encoded_rgb.tif'
base_expr = 'rint(clip((A*100)+10000,0,16777215)).astype(int64)'
subprocess.run([
    'gdal_calc.py',
    '-A', str(cog_tif),
    '--calc', f'bitwise_and({base_expr},255)',
    '--calc', f'bitwise_and(right_shift({base_expr},8),255)',
    '--calc', f'bitwise_and(right_shift({base_expr},16),255)',
    '--type', 'Byte', '--NoDataValue', '0', '--overwrite', '--outfile', str(value_encoded_tif)
], check=True)

value_tiles_dir.mkdir(parents=True, exist_ok=True)
subprocess.run(['gdal2tiles.py', '-r', 'near', '-z', '8-15', '--xyz', '-w', 'none', str(value_encoded_tif), str(value_tiles_dir)], check=True)
print('Value tiles written to:', value_tiles_dir)


Input file size is 1057, 1447
0...10...20...30...40...50...60...70...80...90...100 - done.


Warning 1: Input dataset has no nodata value. Ignoring 'nv' entry in color palette


0...10...20...30...40...50...60...70...80...90...100 - done.


Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done.
0.

...

..10...20...30...40...50...60...70...80...90...100 - done in 00:00:08.
Visual tiles written to: out/annual_climatology_daytime_lst_c/tiles_visual
0...10...20...30...40...50...60...70...80...90...100 - done.


<string>:1: RuntimeWarning: invalid value encountered in cast
Generating Base Tiles:


0...10...20...30...40...50...60...70...80...90...100 - done in 00:00:05.
0..

Generating Overview Tiles:


.10...20...30...40...50...60...70...80...90...100 - done.
Value tiles written to: out/annual_climatology_daytime_lst_c/tiles_values
